# 🔍 Query Expansion for Better Retrieval

**Improve recall by generating query variations**

---

## 📋 Overview

**What you'll learn:**
- Why query expansion works
- Synonym-based expansion
- LLM-based query generation
- Multi-query retrieval
- Fusion strategies

**Time estimate:** ⏱️ 45 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List, Dict
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why Query Expansion?

### Problem:
```
User query: "How do I fix Python errors?"
Document: "Debugging Python exceptions and bugs"

"fix" ≠ "debugging"
"errors" ≠ "exceptions"
→ Might miss relevant doc!
```

### Solution: Expand Query
```
Original: "How do I fix Python errors?"

Expanded:
- "How do I debug Python exceptions?"
- "Fixing Python bugs and errors"
- "Troubleshooting Python problems"

→ Better chance of matching!
```

### Benefits:
- 📈 **Higher recall**: Find more relevant docs
- 🎯 **Vocabulary matching**: Handle synonyms
- 🔍 **Perspective variety**: Different angles
- 💪 **Robustness**: Less sensitive to phrasing

## 💬 LLM-Based Query Expansion

In [ ]:
def expand_query_llm(query: str, num_expansions: int = 3) -> List[str]:
    """Generate query variations using LLM."""
    
    prompt = f"""Generate {num_expansions} different ways to ask the following question.
Use different vocabulary but keep the same meaning.

Original question: {query}

Variations (one per line):"""
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=150
    )
    
    variations_text = response.choices[0].message.content.strip()
    
    # Parse variations (split by newlines, clean up)
    variations = []
    for line in variations_text.split('\n'):
        line = line.strip()
        # Remove numbering if present
        if line and line[0].isdigit():
            line = line.split('. ', 1)[-1] if '. ' in line else line.split(') ', 1)[-1]
        if line:
            variations.append(line)
    
    return [query] + variations[:num_expansions]

# Test
original_query = "How do I install Python packages?"

print(f"🔍 Original Query: {original_query}\n")
print("Expanded Queries:")

expanded = expand_query_llm(original_query, num_expansions=3)
for i, q in enumerate(expanded, 1):
    print(f"  {i}. {q}")

## 🔄 Multi-Query Retrieval

In [ ]:
class MultiQueryRetriever:
    """Retriever using query expansion."""
    
    def __init__(self, documents: List[str], model_name: str = 'all-MiniLM-L6-v2'):
        self.documents = documents
        self.model = SentenceTransformer(model_name)
        self.doc_embeddings = self.model.encode(documents)
    
    def retrieve_single(self, query: str, top_k: int = 5) -> List[Dict]:
        """Standard single-query retrieval."""
        query_emb = self.model.encode([query])[0]
        
        similarities = np.dot(self.doc_embeddings, query_emb) / (
            np.linalg.norm(self.doc_embeddings, axis=1) * np.linalg.norm(query_emb)
        )
        
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        return [{
            'doc_id': int(idx),
            'document': self.documents[idx],
            'score': float(similarities[idx])
        } for idx in top_indices]
    
    def retrieve_multi(
        self,
        original_query: str,
        top_k: int = 5,
        num_expansions: int = 3,
        fusion_method: str = 'rrf'  # 'rrf' or 'max'
    ) -> List[Dict]:
        """Multi-query retrieval with fusion."""
        
        # Expand query
        queries = expand_query_llm(original_query, num_expansions)
        
        # Retrieve for each query
        all_results = []
        for query in queries:
            results = self.retrieve_single(query, top_k=top_k * 2)
            all_results.append(results)
        
        # Fuse results
        if fusion_method == 'rrf':
            return self._fuse_rrf(all_results, top_k)
        elif fusion_method == 'max':
            return self._fuse_max(all_results, top_k)
        else:
            raise ValueError(f"Unknown fusion method: {fusion_method}")
    
    def _fuse_rrf(self, results_list: List[List[Dict]], top_k: int, k: int = 60) -> List[Dict]:
        """Reciprocal Rank Fusion."""
        scores = {}
        
        for results in results_list:
            for rank, result in enumerate(results, 1):
                doc_id = result['doc_id']
                if doc_id not in scores:
                    scores[doc_id] = {'doc': result['document'], 'score': 0}
                scores[doc_id]['score'] += 1 / (k + rank)
        
        # Sort by score
        sorted_results = sorted(
            [(doc_id, data) for doc_id, data in scores.items()],
            key=lambda x: x[1]['score'],
            reverse=True
        )[:top_k]
        
        return [{
            'doc_id': doc_id,
            'document': data['doc'],
            'score': data['score']
        } for doc_id, data in sorted_results]
    
    def _fuse_max(self, results_list: List[List[Dict]], top_k: int) -> List[Dict]:
        """Max score fusion."""
        scores = {}
        
        for results in results_list:
            for result in results:
                doc_id = result['doc_id']
                if doc_id not in scores:
                    scores[doc_id] = {'doc': result['document'], 'score': 0}
                scores[doc_id]['score'] = max(scores[doc_id]['score'], result['score'])
        
        sorted_results = sorted(
            [(doc_id, data) for doc_id, data in scores.items()],
            key=lambda x: x[1]['score'],
            reverse=True
        )[:top_k]
        
        return [{
            'doc_id': doc_id,
            'document': data['doc'],
            'score': data['score']
        } for doc_id, data in sorted_results]

# Test
documents = [
    "Use pip install to add Python packages to your environment.",
    "Python package management is handled by pip and conda.",
    "Install dependencies using pip install package-name command.",
    "Virtual environments isolate Python package installations.",
    "JavaScript uses npm for package management.",
]

retriever = MultiQueryRetriever(documents)

query = "How to add Python libraries?"

print("\n🔍 Comparison: Single vs Multi-Query\n")
print("="*70)

# Single query
print("\n1. Single Query Retrieval:")
single_results = retriever.retrieve_single(query, top_k=3)
for i, r in enumerate(single_results, 1):
    print(f"  {i}. (score: {r['score']:.3f})")
    print(f"     {r['document']}")

# Multi query
print("\n2. Multi-Query Retrieval (with expansion):")
multi_results = retriever.retrieve_multi(query, top_k=3, num_expansions=2)
for i, r in enumerate(multi_results, 1):
    print(f"  {i}. (RRF score: {r['score']:.4f})")
    print(f"     {r['document']}")

## ✅ Summary

### Query Expansion Techniques:

**1. LLM-Based** (Best!)
```python
# Generate semantic variations
"How to install packages?" →
  - "How do I add libraries?"
  - "Installing dependencies in Python"
  - "Package management setup"
```

**2. Synonym-Based**
```python
# Replace words with synonyms
"fix errors" → "resolve bugs", "debug issues"
```

**3. Template-Based**
```python
# Use question templates
"Python packages" →
  - "What are Python packages?"
  - "How to use Python packages?"
  - "Python packages explained"
```

### When to Use:

✅ **Use query expansion when:**
- Recall is more important than precision
- Short user queries (< 5 words)
- Domain-specific vocabulary
- Can afford extra latency (50-200ms)

❌ **Skip when:**
- Need fast responses (< 100ms)
- Queries are already detailed
- High precision required
- Limited compute budget

### Fusion Strategies:

**Reciprocal Rank Fusion (RRF):**
```python
score = Σ 1/(k + rank)
# Combines rankings, not scores
# Robust, works well
```

**Max Score:**
```python
score = max(scores across queries)
# Simple, fast
# Can miss consensus results
```

### Typical Results:

```
Single query:     Recall@5 = 0.70
Multi-query (3):  Recall@5 = 0.85  (+21%)
Multi-query (5):  Recall@5 = 0.90  (+29%)

Latency increase: +50-150ms per expansion
```

### Production Tips:

1. **Cache expansions**
   ```python
   # Cache common query expansions
   cache[query] = expansions
   ```

2. **Limit expansions**
   ```python
   # 2-3 is usually enough
   num_expansions = 3
   ```

3. **Parallel retrieval**
   ```python
   # Retrieve all queries concurrently
   await asyncio.gather(*[retrieve(q) for q in queries])
   ```

### Next Steps:

You've completed the RAG Systems module!

**Next module options:**
- `06_fine_tuning/` - Fine-tuning LLMs
- `07_agents_tools/` - Building AI agents
- `08_production_apis/` - Production deployment